In [ ]:
# Đưa thư mục làm việc về gốc dự án (notebook nằm trong src/)
import os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
print("Thư mục làm việc:", os.getcwd())

In [1]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


# Bước 1: hiểu bài toán và nạp dữ liệu

In [2]:
import pandas as pd
import re

df_wide = pd.read_csv("data/raw/Scan_data_2023_2026.csv")
df_wide['ts'] = pd.to_datetime(df_wide['Date'])
print(f"File gốc: {df_wide.shape[0]} dòng, {df_wide.shape[1]} cột")

rename_variable = {
    'Precipitation Increment (in)':          'prcp',
    'Air Temperature Observed (degF)':       'tair',
    'Wind Speed Maximum (mph)':              'ws',
    'Soil Temperature Observed -2in (degF)': 'st2',
    'Soil Moisture Percent -2in (pct)':      'sm2',
    'Soil Moisture Percent -4in (pct)':      'sm4',
    'Soil Moisture Percent -8in (pct)':      'sm8',
    'Soil Moisture Percent -20in (pct)':     'sm20',
}

column_info = []
for column_name in df_wide.columns:
    match = re.match(r'^(.*?) \((\d+)\) (.*)$', column_name)
    if match:
        station_name = match.group(1)
        station_id   = match.group(2)
        variable     = match.group(3)
        station_full = f'{station_name} ({station_id})'
        column_info.append((station_full, variable, column_name))
print(f"Đã phân tích được {len(column_info)} cột biến-trạm")

all_stations = sorted(set(info[0] for info in column_info))
print(f"Tìm thấy {len(all_stations)} trạm")

long_tables = []
for station in all_stations:
    variable_to_column = {
        variable: column
        for st, variable, column in column_info
        if st == station
    }
    sub_table = df_wide[['ts']].copy()
    sub_table['station'] = station
    for long_name, short_name in rename_variable.items():
        if long_name in variable_to_column:
            original_column = variable_to_column[long_name]
            sub_table[short_name] = df_wide[original_column]
    long_tables.append(sub_table)

df_long = pd.concat(long_tables, ignore_index=True)
df_long = df_long[['station', 'ts',
                   'sm2', 'sm4', 'sm8', 'sm20',
                   'st2', 'prcp', 'tair', 'ws']]

print("\n" + "=" * 50)
print("KẾT QUẢ CUỐI CÙNG")
print("=" * 50)

print(f"\nKích thước bảng long: {df_long.shape}")
print(f"Số trạm: {df_long['station'].nunique()}")
print(f"Khoảng thời gian: {df_long['ts'].min()} → {df_long['ts'].max()}")

print(f"\n5 dòng đầu:")
print(df_long.head())

print(f"\nKiểm tra thiếu dữ liệu (% NaN mỗi cột):")
print((df_long.isna().mean() * 100).round(2))

File gốc: 32160 dòng, 178 cột
Đã phân tích được 176 cột biến-trạm
Tìm thấy 22 trạm

KẾT QUẢ CUỐI CÙNG

Kích thước bảng long: (707520, 10)
Số trạm: 22
Khoảng thời gian: 2023-01-01 00:00:00 → 2026-09-01 23:00:00

5 dòng đầu:
         station                  ts   sm2   sm4   sm8  sm20   st2  prcp  tair    ws
0  Abrams (2092) 2023-01-01 00:00:00  12.4  20.6  25.1  41.7  40.0   0.0  43.7   5.2
1  Abrams (2092) 2023-01-01 01:00:00  12.4  20.6  25.1  41.7  40.0   0.0  43.3   4.5
2  Abrams (2092) 2023-01-01 02:00:00  12.2  20.4  25.1  41.7  39.0   0.0  37.0   6.4
3  Abrams (2092) 2023-01-01 03:00:00  12.2  20.6  25.1  41.7  39.0   0.0  41.7  11.4
4  Abrams (2092) 2023-01-01 04:00:00  12.2  20.4  25.1  41.7  39.0   0.0  37.0  10.4

Kiểm tra thiếu dữ liệu (% NaN mỗi cột):
station     0.00
ts          0.00
sm2        11.01
sm4        14.97
sm8        13.67
sm20       17.52
st2         8.31
prcp        1.43
tair        3.37
ws          1.44
dtype: float64


In [3]:
import os
os.makedirs('report', exist_ok=True)

df = df_long

from ydata_profiling import ProfileReport

sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

print(f"Xong! df shape: {df.shape}")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 487.76it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Xong! df shape: (707520, 10)


# Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

In [4]:
import numpy as np
import pandas as pd

soil_moisture_columns = ['sm2', 'sm4', 'sm8', 'sm20']

for column in soil_moisture_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
    invalid_mask = (df[column] < 0) | (df[column] > 60)
    df.loc[invalid_mask, column] = np.nan
value_columns = ['sm2', 'sm4', 'sm8', 'sm20', 'st2', 'prcp', 'tair', 'ws']

df_hourly = (
    df
    .drop_duplicates(subset=['station', 'ts'])
    .set_index('ts')
    .groupby('station')[value_columns]
    .resample('1h').mean()
    .reset_index()
)

df_hourly[soil_moisture_columns] = (
    df_hourly
    .groupby('station')[soil_moisture_columns]
    .transform(lambda group: group.interpolate(limit=2))
)

df_hourly.to_parquet("data/processed/feat.parquet", index=False)

print(f"Kích thước sau khi làm sạch: {df_hourly.shape}")

Kích thước sau khi làm sạch: (707520, 10)


In [5]:
def apply_cleaning_and_log(df_input, cleaning_steps):

    log_rows = []
    df_current = df_input.copy()
    for step_name, cleaning_function, reason in cleaning_steps:
        rows_before = len(df_current)
        df_current = cleaning_function(df_current)
        rows_after = len(df_current)
        rows_dropped = rows_before - rows_after
        log_rows.append([
            step_name,
            rows_before,
            rows_after,
            rows_dropped,
            reason,
        ])
    log_df = pd.DataFrame(
        log_rows,
        columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason']
    )
    return df_current, log_df
cleaning_steps = [
    (
        'drop duplicates',
        lambda x: x.drop_duplicates(['station', 'ts']),
        'same unit and timestamp'
    ),
    (
        'drop missing target',
        lambda x: x.dropna(subset=['sm8']),
        'cannot be forecast'
    ),
]
df_cleaned, cleaning_log = apply_cleaning_and_log(df_hourly, cleaning_steps)

import os
os.makedirs('report', exist_ok=True)

cleaning_log.to_csv("report/table_cleaning_log.csv", index=False)

print("NHẬT KÝ LÀM SẠCH")
print("=" * 60)
print(cleaning_log)
print("=" * 60)
print(f"\nBảng đã làm sạch: {df_cleaned.shape}")
print(f"Log lưu tại: report/table_cleaning_log.csv")

NHẬT KÝ LÀM SẠCH
                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates       707520      707520        0  same unit and timestamp
1  drop missing target       707520      608379    99141       cannot be forecast

Bảng đã làm sạch: (608379, 10)
Log lưu tại: report/table_cleaning_log.csv


In [6]:
import io
import os
import pandas as pd

os.makedirs('report', exist_ok=True)

if 'df_cleaned' in dir():
    eda_data = df_cleaned.copy()
    print("→ Dùng df_cleaned")
elif 'df_hourly' in dir():
    eda_data = df_hourly.copy()
    print("→ Dùng df_hourly")
elif 'h' in dir():
    eda_data = h.copy()
    print("→ Dùng h")
else:
    raise NameError(
        "Không tìm thấy bảng dữ liệu. "
        "Chạy các cell làm sạch trước."
    )

print(f"Kích thước: {eda_data.shape}\n")

buffer = io.StringIO()
eda_data.info(buf=buffer)
info_text = buffer.getvalue()

print("THÔNG TIN CẤU TRÚC")
print("=" * 60)
print(info_text[:3000])

describe_numeric = eda_data.describe().T
describe_numeric['skew'] = eda_data.select_dtypes('number').skew()
describe_numeric = describe_numeric.round(3)

describe_numeric.to_csv("report/table_describe_numeric.csv")

print("\nTHỐNG KÊ CỘT SỐ")
print("=" * 60)
print(describe_numeric.head(15))

categorical_columns = eda_data.select_dtypes(exclude='number')

if categorical_columns.shape[1] > 0:
    describe_categorical = categorical_columns.describe().T
    describe_categorical.to_csv("report/table_describe_categorical.csv")

    print("\nTHỐNG KÊ CỘT PHÂN LOẠI")
    print("=" * 60)
    print(describe_categorical.head(15))
else:
    print("\nKhông có cột phân loại nào.")

→ Dùng df_cleaned
Kích thước: (608379, 10)

THÔNG TIN CẤU TRÚC
<class 'pandas.core.frame.DataFrame'>
Index: 608379 entries, 0 to 707519
Data columns (total 10 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   station  608379 non-null  object        
 1   ts       608379 non-null  datetime64[ns]
 2   sm2      589011 non-null  float64       
 3   sm4      574971 non-null  float64       
 4   sm8      608379 non-null  float64       
 5   sm20     558431 non-null  float64       
 6   st2      599154 non-null  float64       
 7   prcp     606912 non-null  float64       
 8   tair     599256 non-null  float64       
 9   ws       606907 non-null  float64       
dtypes: datetime64[ns](1), float64(8), object(1)
memory usage: 51.1+ MB


THỐNG KÊ CỘT SỐ
         count                           mean                  min                  25%                  50%                  75%                  max  \
ts      608379  2024-12-05 15:11:43

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('report', exist_ok=True)

if 'df_cleaned' in dir():
    eda_data = df_cleaned.copy()
    print("→ Dùng df_cleaned")
elif 'df_hourly' in dir():
    eda_data = df_hourly.copy()
    print("→ Dùng df_hourly")
elif 'h' in dir():
    eda_data = h.copy()
    print("→ Dùng h")
else:
    raise NameError("Không tìm thấy bảng dữ liệu. Chạy các cell làm sạch trước.")

print(f"Kích thước: {eda_data.shape}\n")

missing_rate = (
    eda_data.isna().mean()
    .sort_values(ascending=False)
    .rename('missing_rate')
    .to_frame()
)

missing_rate['decision'] = np.select(
    condlist=[
        missing_rate['missing_rate'] == 0,
        missing_rate['missing_rate'] < 0.05,
        missing_rate['missing_rate'] < 0.40,
    ],
    choicelist=[
        'keep',
        'interpolate/impute median',
        'drop or keep flag',
    ],
    default='drop column'
)

missing_rate = missing_rate.round(4)
missing_rate.to_csv("report/table_missing.csv")

print("TỶ LỆ THIẾU TỪNG CỘT")
print("=" * 60)
print(missing_rate.head(15))

time_column = 'ts'
station_column = 'station'

timestamps = pd.to_datetime(eda_data[time_column], errors='coerce')
month_labels = timestamps.dt.to_period('M')

missing_by_time = (
    eda_data.groupby(month_labels)['sm8']
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
)

print("\nTHÁNG THIẾU sm8 NHIỀU NHẤT")
print("=" * 60)
print(missing_by_time.head(5).round(3))

missing_by_station = (
    eda_data.groupby(station_column)['sm8']
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
)

print("\nTRẠM THIẾU sm8 NHIỀU NHẤT")
print("=" * 60)
print(missing_by_station.head(5).round(3))

nan_flags = eda_data.isna().astype(int)

columns_with_missing = nan_flags.columns[nan_flags.sum() > 0]
nan_flags = nan_flags[columns_with_missing]

if nan_flags.shape[1] > 1:
    co_missing_corr = nan_flags.corr().round(2)
    print("\nMA TRẬN ĐỒNG THIẾU (tương quan giữa các cột NaN-flag)")
    print("=" * 60)
    print(co_missing_corr)

fig, ax = plt.subplots(figsize=(7, 3))
missing_rate['missing_rate'].head(12).plot.bar(ax=ax, color='#1B6B6D')
ax.set_ylabel('Missing rate')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig("report/fig_eda_missing.png", dpi=300)
plt.show()

n_zero = (eda_data['sm8'] == 0).sum()
n_positive = (eda_data['sm8'] > 0).sum()

print("\nBẤT THƯỜNG Ở sm8")
print("=" * 60)
print(f"sm8 = 0 : {n_zero} dòng")
print(f"sm8 > 0 : {n_positive} dòng")
if n_zero > 0:
    print(f"Tỷ lệ sm8=0: {n_zero / len(eda_data) * 100:.2f}% — kiểm tra xem có phải cảm biến báo lỗi không.")

print(f"\nĐã lưu:")
print("  - report/table_missing.csv")
print("  - report/fig_eda_missing.png")

→ Dùng df_cleaned
Kích thước: (608379, 10)

TỶ LỆ THIẾU TỪNG CỘT
         missing_rate                   decision
sm20           0.0821          drop or keep flag
sm4            0.0549          drop or keep flag
sm2            0.0318  interpolate/impute median
st2            0.0152  interpolate/impute median
tair           0.0150  interpolate/impute median
ws             0.0024  interpolate/impute median
prcp           0.0024  interpolate/impute median
station        0.0000                       keep
ts             0.0000                       keep
sm8            0.0000                       keep

THÁNG THIẾU sm8 NHIỀU NHẤT
ts
2023-01    0.0
2024-12    0.0
2025-02    0.0
2025-03    0.0
2025-04    0.0
Freq: M, Name: sm8, dtype: float64

TRẠM THIẾU sm8 NHIỀU NHẤT
station
Abrams (2092)          0.0
Beaumont (2207)        0.0
Vernon (2202)          0.0
Uvalde (2204)          0.0
Stephenville (2203)    0.0
Name: sm8, dtype: float64

MA TRẬN ĐỒNG THIẾU (tương quan giữa các cột NaN-flag)
    

In [8]:
import os
import numpy as np
import pandas as pd

os.makedirs('report', exist_ok=True)

if 'df_cleaned' in dir():
    eda_data = df_cleaned.copy()
    print("→ Dùng df_cleaned")
elif 'df_hourly' in dir():
    eda_data = df_hourly.copy()
    print("→ Dùng df_hourly")
elif 'h' in dir():
    eda_data = h.copy()
    print("→ Dùng h")
else:
    raise NameError("Không tìm thấy bảng dữ liệu.")

candidate_columns = ['sm2', 'sm4', 'sm8', 'sm20', 'st2', 'prcp', 'tair', 'ws']
numeric_columns = [c for c in candidate_columns if c in eda_data.columns]

correlation_matrix = eda_data[numeric_columns].corr()

correlation_matrix.round(3).to_csv("report/table_corr.csv")

print("MA TRẬN TƯƠNG QUAN")
print("=" * 60)
print(correlation_matrix.round(3))

column_decisions = []

for column in eda_data.columns:

    if column == 'sm8':
        column_decisions.append([column, 'target', 'keep'])
        continue

    if column in ('ts', 'station'):
        column_decisions.append([column, 'time/unit key', 'keep for feature building'])
        continue

    if column in numeric_columns:
        if 'sm8' in correlation_matrix.columns:
            r = correlation_matrix.loc[column, 'sm8']
        else:
            r = np.nan

        reason = f'correlation with target {r:.2f}'
        decision = 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'
        column_decisions.append([column, reason, decision])

    else:
        column_decisions.append([column, 'other', 'review'])

column_decisions_df = pd.DataFrame(
    column_decisions,
    columns=['column', 'reason', 'decision']
)
column_decisions_df.to_csv("report/table_columns.csv", index=False)

print("\nĐỀ XUẤT GIỮ/BỎ CỘT")
print("=" * 60)
print(column_decisions_df)

def summary_statistics(series):
    """Trả về Series chứa 5 chỉ số tóm tắt."""
    return pd.Series({
        'mean':   series.mean(),
        'median': series.median(),
        'p05':    series.quantile(0.05),
        'p95':    series.quantile(0.95),
        'n':      series.count(),
    })

target_by_station = (
    eda_data.groupby('station')['sm8']
    .apply(summary_statistics)
    .unstack()
    .sort_values('mean', ascending=False)
)

target_by_station.round(3).to_csv("report/table_target_by_unit.csv")

print("\nTOP 10 TRẠM CÓ sm8 TRUNG BÌNH CAO NHẤT")
print("=" * 60)
print(target_by_station.head(10).round(3))

if np.issubdtype(eda_data['ts'].dtype, np.datetime64):
    month_numbers = eda_data['ts'].dt.month
else:
    month_numbers = pd.to_datetime(eda_data['ts'], errors='coerce').dt.month

target_by_month = (
    eda_data.groupby(month_numbers)['sm8']
    .apply(summary_statistics)
    .unstack()
)

target_by_month.round(3).to_csv("report/table_target_by_period.csv")

print("\nsm8 THEO THÁNG (1-12)")
print("=" * 60)
print(target_by_month.round(3))

print(f"\nĐã lưu:")
print("  - report/table_corr.csv")
print("  - report/table_columns.csv")
print("  - report/table_target_by_unit.csv")
print("  - report/table_target_by_period.csv")

→ Dùng df_cleaned
MA TRẬN TƯƠNG QUAN
        sm2    sm4    sm8   sm20    st2   prcp   tair     ws
sm2   1.000  0.891  0.780  0.663 -0.174  0.045 -0.110 -0.062
sm4   0.891  1.000  0.887  0.716 -0.188  0.031 -0.130 -0.040
sm8   0.780  0.887  1.000  0.774 -0.197  0.020 -0.138 -0.044
sm20  0.663  0.716  0.774  1.000 -0.098  0.019 -0.042 -0.099
st2  -0.174 -0.188 -0.197 -0.098  1.000  0.019  0.901 -0.021
prcp  0.045  0.031  0.020  0.019  0.019  1.000  0.009  0.090
tair -0.110 -0.130 -0.138 -0.042  0.901  0.009  1.000  0.108
ws   -0.062 -0.040 -0.044 -0.099 -0.021  0.090  0.108  1.000

ĐỀ XUẤT GIỮ/BỎ CỘT
    column                         reason                   decision
0  station                  time/unit key  keep for feature building
1       ts                  time/unit key  keep for feature building
2      sm2   correlation with target 0.78                       keep
3      sm4   correlation with target 0.89                       keep
4      sm8                         target        

### Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

In [9]:
import os
import duckdb
os.makedirs('report', exist_ok=True)
os.makedirs('sql', exist_ok=True)

if 'con' not in globals():
    con = duckdb.connect("data/processed/ady.duckdb")
SQL_SCRIPT = r"""

SELECT
    station,
    month(ts)     AS mon,
    AVG(sm2)      AS sm2,
    AVG(sm8)      AS sm8,
    AVG(sm20)     AS sm20,
    SUM((sm8 < 10)::INT) AS dry_hours
FROM sc
GROUP BY 1, 2
ORDER BY 1, 2;

SELECT
    CASE
        WHEN r24 > 10 THEN 'heavy'
        WHEN r24 > 0  THEN 'light'
        ELSE 'none'
    END AS rain24,
    AVG(sm2 - sm2_lag24) AS dsm2
FROM (
    SELECT
        *,
        SUM(prcp) OVER (
            PARTITION BY station
            ORDER BY ts
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        ) AS r24,
        LAG(sm2, 24) OVER (PARTITION BY station ORDER BY ts) AS sm2_lag24
    FROM sc
)
GROUP BY 1;


CREATE OR REPLACE TABLE feat AS
SELECT
    station,
    ts,
    sm2, sm4, sm8, sm20, st2, prcp, tair, ws,
    hour(ts)  AS hr,
    month(ts) AS mon,

    -- Lag features (quá khứ)
    LAG(sm8, 24)  OVER w AS sm8_lag24,
    LAG(sm8, 168) OVER w AS sm8_lag7d,
    LAG(sm8, 1)   OVER w AS sm8_lag1,
    LAG(sm8, 6)   OVER w AS sm8_lag6,

    -- Delta features (rate of change gần đây)
    sm8 - LAG(sm8, 24)  OVER w AS sm8_delta24,
    sm8 - LAG(sm8, 6)   OVER w AS sm8_delta6,

    -- Rolling sum/avg (quá khứ)
    SUM(prcp) OVER (w ROWS BETWEEN 23  PRECEDING AND CURRENT ROW) AS r24,
    SUM(prcp) OVER (w ROWS BETWEEN 167 PRECEDING AND CURRENT ROW) AS r7d,
    AVG(tair) OVER (w ROWS BETWEEN 71  PRECEDING AND CURRENT ROW) AS t3d,

    -- Target (tương lai)
    LEAD(sm8, 24) OVER w AS y_24h,
    LEAD(sm8, 72) OVER w AS y_72h

FROM sc
WINDOW w AS (PARTITION BY station ORDER BY ts);

SELECT
    COUNT(*)     AS n,
    COUNT(y_72h) AS n72
FROM feat;
"""

lines_without_comments = []

for line in SQL_SCRIPT.splitlines():
    if not line.strip().startswith('--'):
        lines_without_comments.append(line)

sql_clean = '\n'.join(lines_without_comments)
sql_statements = sql_clean.split(';')
query_counter = 0

for statement in sql_statements:
    statement = statement.strip()
    if not statement:
        continue
    result = con.execute(statement)
    if statement.upper().startswith(('SELECT', 'WITH')):
        query_counter += 1
        result_df = result.df()
        output_path = f'report/table_q{query_counter}.csv'
        result_df.to_csv(output_path, index=False)
        print(f'\n--- Q{query_counter} ---')
        print(result_df.head(20))

with open('sql/queries.sql', 'w') as f:
    f.write(SQL_SCRIPT)

print(f"\n\nĐã chạy {query_counter} truy vấn.")
print(f"SQL script lưu tại: sql/queries.sql")


--- Q1 ---
            station  mon        sm2        sm8       sm20  dry_hours
0     Abrams (2092)    1  13.792204  24.599149  39.073101        0.0
1     Abrams (2092)    2  14.570612  25.277010  39.534338        0.0
2     Abrams (2092)    3  12.613239  23.189651  39.461910        0.0
3     Abrams (2092)    4  11.004340  19.953519  39.130787      125.0
4     Abrams (2092)    5  12.063491  22.030809  39.223705      275.0
5     Abrams (2092)    6  14.581904  24.871214  39.702733        0.0
6     Abrams (2092)    7  11.894523  21.277621  37.439751       83.0
7     Abrams (2092)    8   7.490289  12.886626  29.169691     1279.0
8     Abrams (2092)    9   9.219277  14.283699  29.052793     1186.0
9     Abrams (2092)   10   8.494400  12.244825  28.230287      831.0
10    Abrams (2092)   11  15.281667  24.986597  33.541574       41.0
11    Abrams (2092)   12  15.063127  25.597917  38.859185        0.0
12  Beaumont (2207)    1  43.993750  40.956852  44.933602        0.0
13  Beaumont (2207)   

In [10]:
import os
import numpy as np
import pandas as pd
from scipy.stats import kruskal, spearmanr

os.makedirs('report', exist_ok=True)

if 'con' not in globals():
    import duckdb
    con = duckdb.connect("data/processed/ady.duckdb")

feat = con.execute("SELECT * FROM feat").df()
print(f"Kích thước bảng feat: {feat.shape}")

if np.issubdtype(feat['ts'].dtype, np.number):
    feat['period'] = feat['ts']
else:
    feat['period'] = pd.to_datetime(feat['ts'], errors='coerce').dt.month

rq1a_by_period = (
    feat.groupby('period')['sm8']
    .agg(['mean', 'median', 'std', 'count'])
    .round(3)
)

rq1a_by_period.to_csv("report/table_rq1a_period.csv")

print("\nRQ1a — sm8 THEO THÁNG")
print("=" * 60)
print(rq1a_by_period)

groups_by_period = [
    values.values
    for _, values in feat.groupby('period')['sm8']
    if len(values) > 30
]

if len(groups_by_period) > 1:
    kruskal_statistic, kruskal_pvalue = kruskal(*groups_by_period)
    print(f"\nKruskal-Wallis: H = {kruskal_statistic:.2f}, p = {kruskal_pvalue:.2e}")

    if kruskal_pvalue < 0.05:
        print("→ Có sự khác biệt có ý nghĩa thống kê giữa các tháng (chu kỳ mùa TỒN TẠI).")
    else:
        print("→ Không đủ bằng chứng để nói các tháng khác nhau.")

external_features = [
    column for column in feat.select_dtypes('number').columns
    if column not in ('sm8', 'y_24h')
    and not column.startswith('sm8')
]

correlation_rows = []

for feature in external_features[:15]:
    r, p_value = spearmanr(feat[feature], feat['sm8'], nan_policy='omit')
    correlation_rows.append([feature, round(r, 3), p_value])

rq1b_correlation = pd.DataFrame(
    correlation_rows,
    columns=['variable', 'spearman_r', 'p_value']
).sort_values('spearman_r', key=abs, ascending=False)

rq1b_correlation.to_csv('report/table_rq1b_corr.csv', index=False)

print("\nRQ1b — TƯƠNG QUAN SPEARMAN VỚI sm8")
print("=" * 60)
print(rq1b_correlation)

rq1c_by_station = (
    feat.groupby('station')['sm8']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
)

rq1c_top_and_bottom = pd.concat([
    rq1c_by_station.head(5),
    rq1c_by_station.tail(5),
]).round(3)

rq1c_top_and_bottom.to_csv("report/table_rq1c_units.csv")

print("\nRQ1c — TOP 5 TRẠM ẨM NHẤT VÀ KHÔ NHẤT")
print("=" * 60)
print(rq1c_top_and_bottom)

month_with_highest_mean = rq1a_by_period['mean'].idxmax()
month_with_lowest_mean = rq1a_by_period['mean'].idxmin()

highest_mean = rq1a_by_period['mean'].max()
lowest_mean = rq1a_by_period['mean'].min()

seasonal_ratio = round(highest_mean / max(lowest_mean, 1e-9), 2)

print(f"\nKẾT LUẬN RQ1")
print("=" * 60)
print(f"Tháng có sm8 cao nhất : tháng {month_with_highest_mean} (mean = {highest_mean:.2f}%)")
print(f"Tháng có sm8 thấp nhất: tháng {month_with_lowest_mean} (mean = {lowest_mean:.2f}%)")
print(f"Tỷ số cao/thấp        : {seasonal_ratio}x")
print(f"\nĐã lưu:")
print("  - report/table_rq1a_period.csv")
print("  - report/table_rq1b_corr.csv")
print("  - report/table_rq1c_units.csv")

Kích thước bảng feat: (707520, 23)

RQ1a — sm8 THEO THÁNG
          mean  median     std  count
period                               
1       26.092    24.4  11.439  54003
2       27.066    26.0  11.474  48897
3       26.703    25.3  12.066  54171
4       25.792    23.8  11.874  51910
5       26.019    24.6  11.436  55688
6       25.764    24.5  11.195  57462
7       24.267    21.8  11.552  59954
8       21.137    18.9  10.594  58570
9       22.277    21.9  10.885  41182
10      20.667    20.4   9.787  42592
11      25.341    24.4  11.695  41843
12      26.490    25.8  11.477  43650

Kruskal-Wallis: H = nan, p = nan
→ Không đủ bằng chứng để nói các tháng khác nhau.

RQ1b — TƯƠNG QUAN SPEARMAN VỚI sm8
   variable  spearman_r        p_value
12    y_72h       0.966   0.000000e+00
1       sm4       0.878   0.000000e+00
2      sm20       0.760   0.000000e+00
0       sm2       0.756   0.000000e+00
10      r7d       0.251   0.000000e+00
3       st2      -0.201   0.000000e+00
11      t3d      

# Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

In [11]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('report', exist_ok=True)

def clean_axes_style(ax):
    """Loại bỏ đường viền không cần thiết ở top và right của biểu đồ."""
    ax.spines[['top', 'right']].set_visible(False)

data = feat.dropna(subset=['sm8'])
print(f"Dữ liệu vẽ: {len(data):,} dòng")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].hist(data['sm8'], bins=50, color='#1B6B6D')
axes[0].set_xlabel('sm8 (%)')
axes[0].set_ylabel('Count')
clean_axes_style(axes[0])

axes[1].boxplot(data['sm8'], vert=False)
axes[1].set_xlabel('sm8 (%)')
clean_axes_style(axes[1])

fig.tight_layout()
fig.savefig("report/fig_rq1_distribution.png", dpi=300)
plt.close(fig)

sample_stations = data['station'].unique()[:3]

fig, ax = plt.subplots(figsize=(9, 3))

for station in sample_stations:
    station_data = data[data['station'] == station].sort_values('ts')
    ax.plot(
        station_data['ts'],
        station_data['sm8'],
        linewidth=0.8,
        label=str(station)
    )

ax.set_ylabel('sm8 (%)')
ax.legend(frameon=False)
clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_rq1_timeseries.png", dpi=300)
plt.close(fig)

top_15_stations = (
    data.groupby('station')['sm8']
    .mean()
    .nlargest(15)
    .index
)
heatmap_data = (
    data[data['station'].isin(top_15_stations)]
    .pivot_table(
        index='station',
        columns='period',
        values='sm8',
        aggfunc='mean'
    )
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    heatmap_data,
    cmap='YlGnBu',
    ax=ax,
    cbar_kws={'label': 'sm8 (%)'}
)
ax.set_xlabel('Period (tháng)')
ax.set_ylabel('Trạm')

fig.tight_layout()
fig.savefig("report/fig_rq1_heatmap.png", dpi=300)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 3))

sns.boxplot(
    data=data,
    x='period',
    y='sm8',
    color='#9FBFBF',
    ax=ax,
    showfliers=False
)
clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_rq1_box_period.png", dpi=300)
plt.close(fig)

if 'rq1b_correlation' in dir() and len(rq1b_correlation) > 0:
    strongest_feature = rq1b_correlation.iloc[0]['variable']

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.scatter(
        data[strongest_feature],
        data['sm8'],
        s=4,
        alpha=0.3,
        color='#1B6B6D'
    )
    ax.set_xlabel(strongest_feature)
    ax.set_ylabel('sm8 (%)')
    clean_axes_style(ax)

    fig.tight_layout()
    fig.savefig("report/fig_rq1_scatter.png", dpi=300)
    plt.close(fig)

correlation = data.select_dtypes('number').corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    correlation,
    cmap='coolwarm',
    center=0,
    ax=ax,
    annot=False
)

fig.tight_layout()
fig.savefig("report/fig_rq1_corr.png", dpi=300)
plt.close(fig)

monthly_means = data.groupby('period')['sm8'].mean()

fig, ax = plt.subplots(figsize=(9, 3))

bars = ax.bar(
    monthly_means.index.astype(str),
    monthly_means.values,
    color='#1B6B6D'
)

ax.bar_label(
    bars,
    fmt='%.1f',
    fontweight='bold',
    fontsize=8
)

ax.set_ylabel('Mean sm8 (%)')
clean_axes_style(ax)

fig.tight_layout()
fig.savefig("report/fig_rq1_period_bar.png", dpi=300)
plt.close(fig)

print("\n7 biểu đồ đã lưu tại report/:")
print("  1. fig_rq1_distribution.png  — histogram + boxplot")
print("  2. fig_rq1_timeseries.png    — chuỗi thời gian 3 trạm")
print("  3. fig_rq1_heatmap.png       — heatmap trạm × tháng")
print("  4. fig_rq1_box_period.png    — boxplot theo tháng")
print("  5. fig_rq1_scatter.png       — scatter với feature mạnh")
print("  6. fig_rq1_corr.png          — heatmap tương quan")
print("  7. fig_rq1_period_bar.png    — bar chart có nhãn")
print("\nCaption gợi ý cho mỗi hình: KẾT LUẬN + SỐ CỤ THỂ + ĐIỀU KIỆN")

Dữ liệu vẽ: 609,922 dòng

7 biểu đồ đã lưu tại report/:
  1. fig_rq1_distribution.png  — histogram + boxplot
  2. fig_rq1_timeseries.png    — chuỗi thời gian 3 trạm
  3. fig_rq1_heatmap.png       — heatmap trạm × tháng
  4. fig_rq1_box_period.png    — boxplot theo tháng
  5. fig_rq1_scatter.png       — scatter với feature mạnh
  6. fig_rq1_corr.png          — heatmap tương quan
  7. fig_rq1_period_bar.png    — bar chart có nhãn

Caption gợi ý cho mỗi hình: KẾT LUẬN + SỐ CỤ THỂ + ĐIỀU KIỆN


## Test case của Notebook_A

In [12]:
import os
import json
import hashlib
import pandas as pd

if 'con' not in globals():
    import duckdb
    con = duckdb.connect("data/processed/ady.duckdb")

feat = con.execute("SELECT * FROM feat").df()
print(f"Kích thước bảng feat: {feat.shape}")

assert len(feat) >= 30000, \
    f'Bảng chỉ có {len(feat)} dòng, cần ít nhất 30,000. Mở rộng phạm vi.'

non_null_targets = feat['y_24h'].notna().sum()
required_targets = 0.8 * len(feat)

assert non_null_targets > required_targets, \
    f'Chỉ có {non_null_targets} target non-null, cần > {required_targets:.0f}.'

key_columns = ['station', 'ts']

assert not feat.duplicated(key_columns).any(), \
    'Có dòng trùng khoá station+ts. Kiểm tra lại bước làm sạch.'

lead_columns = [
    column for column in feat.columns
    if column.startswith('y_') and column != 'y_24h'
]

assert all(column.startswith('y_') for column in lead_columns), \
    'Có cột feature chứa giá trị tương lai (LEAD). Rò rỉ!'

print('Tests A: OK — bảng đã qua 4 kiểm tra chất lượng')

output_path = "data/processed/feat.parquet"
os.makedirs('data/processed', exist_ok=True)

feat.to_parquet(output_path, index=False)

with open(output_path, 'rb') as f:
    file_bytes = f.read()
    md5_hash = hashlib.md5(file_bytes).hexdigest()

manifest = {
    'rows':     int(len(feat)),
    'cols':     list(feat.columns),
    'md5':      md5_hash,
    'time_min': str(feat['ts'].min()),
    'time_max': str(feat['ts'].max()),
    'author':   'Student A',
}

manifest_path = 'data/processed/manifest.json'

with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)

print('\nMANIFEST bàn giao cho Notebook B:')
print(json.dumps(manifest, indent=2, default=str))
print(f"\nBảng feature lưu tại : {output_path}")
print(f"Manifest lưu tại      : {manifest_path}")

Kích thước bảng feat: (707520, 23)
Tests A: OK — bảng đã qua 4 kiểm tra chất lượng

MANIFEST bàn giao cho Notebook B:
{
  "rows": 707520,
  "cols": [
    "station",
    "ts",
    "sm2",
    "sm4",
    "sm8",
    "sm20",
    "st2",
    "prcp",
    "tair",
    "ws",
    "hr",
    "mon",
    "sm8_lag24",
    "sm8_lag7d",
    "sm8_lag1",
    "sm8_lag6",
    "sm8_delta24",
    "sm8_delta6",
    "r24",
    "r7d",
    "t3d",
    "y_24h",
    "y_72h"
  ],
  "md5": "8f965b616194e9c7652c7dd961a7c1b8",
  "time_min": "2023-01-01 00:00:00",
  "time_max": "2026-09-01 23:00:00",
  "author": "Student A"
}

Bảng feature lưu tại : D:/ADYm/data/processed/feat.parquet
Manifest lưu tại      : data/processed/manifest.json
